# Hyperparameter tuning for the candidate model

This notebook tunes only `hist_gradient_boosting`, the winner of the baseline comparison. The search uses out-of-fold predictions inside the `fit` block, without a threshold, without calibration, and without consulting the official test.

## 1. Tuning protocol

The question is whether a different hyperparameter configuration improves probability quality before calibration and threshold selection. The search is a small explicit grid, compared with group-aware OOF PR-AUC, ROC-AUC, and Brier.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from adult_data import load_adult
from hyperparameter_tuning import write_hgb_tuning_outputs

data = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
tuning_dir = ROOT / 'data' / 'processed' / 'tuning'
outputs = write_hgb_tuning_outputs(data, tuning_dir)
outputs

{'metrics': WindowsPath('D:/Projetos/ModelRisk_ResponsibleML/data/processed/tuning/hgb_tuning_metrics.csv'),
 'contract': WindowsPath('D:/Projetos/ModelRisk_ResponsibleML/data/processed/tuning/hgb_tuning_contract.json')}

## 2. OOF results

The metrics below evaluate out-of-fold probabilities in the `fit` block. None depends on choosing an operational threshold; that decision remains in Notebook 5 after calibration.

In [2]:
metrics = pd.read_csv(tuning_dir / 'hgb_tuning_metrics.csv')
contract = json.loads((tuning_dir / 'hgb_tuning_contract.json').read_text(encoding='utf-8'))
display(metrics.round(4))
display(pd.Series(contract, name='value').to_frame())

,candidate_id,max_iter,learning_rate,max_leaf_nodes,min_samples_leaf,roc_auc,pr_auc,brier
0,conservative_leaves,250,0.05,15,30,0.9233,0.8188,0.0910
1,baseline_fixed,200,0.08,31,20,0.9234,0.8184,0.0910
2,deeper_regularized,300,0.05,31,50,0.9226,0.8176,0.0914
3,shallow_regularized,300,0.08,15,50,0.9228,0.8171,0.0914
4,faster_medium,150,0.10,31,20,0.9227,0.8166,0.0915
5,wide_faster,200,0.12,63,20,0.9216,0.8149,0.0923


,value
protocol_version,hgb-tuning-mvp-1
tuned_model,hist_gradient_boosting
search_method,explicit_candidate_grid
candidate_count,6
candidate_ids,"[baseline_fixed, conservative_leaves, deeper_r..."
objective_order,"[pr_auc, roc_auc, brier]"
fit_rows,19537
oof_folds,3
group_aware,True
random_state,44


### Insights:
The winning configuration is recorded by highest OOF PR-AUC, with ROC-AUC and Brier as tie-breakers. The threshold does not participate in this choice. The result is the best configuration within the candidate grid, not an exhaustive search over every possible hyperparameter.

## 3. Handoff to final selection

The tuning contract supplies the selected hyperparameters to Notebook 5. There, the pipeline is fitted in `fit`, calibrated in `calibration`, and evaluated across the threshold grid in `selection`. The official test remains reserved.